In [10]:
%uv pip install pyarrow
%uv pip install fastparquet
%uv pip install duckdb
%uv pip install plotly

Using Python 3.11.12 environment at: /usr/local
Resolved 1 package in 166ms                                          
Prepared 1 package in 1.25s                                              
Installed 1 package in 58ms                                 
 + pyarrow==26.0.0
Note: you may need to restart the kernel to use updated packages.
Using Python 3.11.12 environment at: /usr/local
Resolved 9 packages in 305ms                                         
Prepared 3 packages in 155ms                                             
Installed 3 packages in 22ms                                
 + cramjam==2.14.0
 + fastparquet==2026.9.0
 + fsspec==2026.9.0
Note: you may need to restart the kernel to use updated packages.
Using Python 3.11.12 environment at: /usr/local
Resolved 1 package in 206ms                                          
Prepared 1 package in 514ms                                              
Installed 1 package in 30ms                                 
 + duckdb==2.0.0
Note: you

In [15]:
import duckdb
import pandas as pd
import plotly.express as px

titles_path = "/mnt/titles/tagged_track_titles.parquet"
track_tags_path = "/mnt/titles/track_tags.parquet"
tracks_path = "/mnt/kaggle/classical_tracks_with_roles.parquet"

pd.set_option("display.max_columns", None)

In [4]:
con = duckdb.connect()

con.execute(f"CREATE OR REPLACE VIEW tagged_titles AS SELECT * FROM read_parquet('{titles_path}')")
con.execute(f"CREATE OR REPLACE VIEW track_tags AS SELECT * FROM read_parquet('{track_tags_path}')")
con.execute(f"CREATE OR REPLACE VIEW classical_tracks AS SELECT * FROM read_parquet('{tracks_path}')")

for name in ["tagged_titles", "track_tags", "classical_tracks"]:
    rows = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"{name}: {rows:,} rows")

tagged_titles: 5,026,460 rows
track_tags: 18,320,983 rows
classical_tracks: 15,417,804 rows


In [7]:
for name in ["tagged_titles", "track_tags", "classical_tracks"]:
    cols = con.execute(f"DESCRIBE {name}").fetchall()
    col_names = [row[0] for row in cols]
    print(f"{name}: {col_names}")

tagged_titles: ['tagged_name_track', 'track_rowid', '__index_level_0__']
track_tags: ['track_rowid', 'type', 'value']
classical_tracks: ['artist_rowid', 'track_rowid', 'id_artist', 'id_track', 'name_artist', 'name_track', 'followers_total', 'popularity_artist', 'popularity_track', 'genre', 'artist_roles']


In [17]:
import plotly.express as px

composer_counts = con.execute("""
SELECT c.name_artist AS composer, COUNT(DISTINCT c.track_rowid) AS tracks
FROM classical_tracks c
WHERE c.artist_roles IS NOT NULL
  AND c.artist_roles <> ''
  AND EXISTS (
      SELECT 1
      FROM unnest(
          from_json(c.artist_roles, '[{"id":"VARCHAR","name":"VARCHAR","role":"VARCHAR"}]')
      ) AS u(r)
      WHERE r.id = c.id_artist
        AND lower(r.role) = 'composer'
  )
GROUP BY composer
ORDER BY tracks DESC
""").df()

plot = composer_counts.head(10).copy()
plot.loc[len(plot)] = {"composer": "other", "tracks": composer_counts["tracks"].iloc[10:].sum()}

fig = px.pie(
    plot,
    names="composer",
    values="tracks",
    hole=0.45,
    color="composer",
    color_discrete_sequence=px.colors.sequential.Teal_r * 2,
    title=f"Composer artists by tagged tracks (top 10 of {len(composer_counts):,})",
)
fig.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hovertemplate="%{label}<br>%{value:,} tracks (%{percent})<extra></extra>",
)
fig.update_layout(showlegend=False, margin=dict(t=60, b=20, l=20, r=20))
fig.show()

In [ ]:
tag_counts = con.execute("""
SELECT type, COUNT(*) AS occurrences
FROM track_tags
GROUP BY type
ORDER BY occurrences DESC
""").df()

fig = px.pie(
    tag_counts,
    names="type",
    values="occurrences",
    hole=0.45,
    color="type",
    color_discrete_sequence=px.colors.sequential.Teal_r,
    title=f"Tag occurrences in track_tags ({tag_counts['occurrences'].sum():,} tags)",
)
fig.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hovertemplate="%{label}<br>%{value:,} tags (%{percent})<extra></extra>",
)
fig.update_layout(showlegend=False, margin=dict(t=60, b=20, l=20, r=20))
fig.show()

In [12]:
catalog_letters = con.execute(r"""
SELECT lower(letters) AS letters, COUNT(*) AS occurrences
FROM (
    SELECT unnest(regexp_extract_all(value, '[\p{L}]+')) AS letters
    FROM track_tags
    WHERE type = 'catalog'
)
GROUP BY letters
ORDER BY occurrences DESC
""").df()

catalog_letters.head(20)

,letters,occurrences
0,op,1144061
1,bwv,298885
2,k,218994
3,no,108416
4,d,80028
5,hwv,74438
6,rv,69385
7,hob,54249
8,a,40225
9,l,38770


In [14]:
plot = catalog_letters.head(10).copy()
plot.loc[len(plot)] = {"letters": "other", "occurrences": catalog_letters["occurrences"].iloc[10:].sum()}

fig = px.pie(
    plot,
    names="letters",
    values="occurrences",
    hole=0.45,
    color="letters",
    color_discrete_sequence=px.colors.sequential.Teal_r * 2,
    title=f"Catalogs tags (top 10 of {len(catalog_letters):,})",
)
fig.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hovertemplate="%{label}<br>%{value:,} occurrences (%{percent})<extra></extra>",
)
fig.update_layout(showlegend=False, margin=dict(t=60, b=20, l=20, r=20))
fig.show()

In [25]:
beethoven_111 = con.execute("""
WITH catalog AS (
    SELECT
        track_rowid,
        value,
        regexp_extract_all(lower(value), '[\p{L}\p{N}]+') AS tokens
    FROM track_tags
    WHERE type = 'catalog'
)
SELECT DISTINCT c.track_rowid, c.name_track, cat.value AS catalog
FROM classical_tracks c
JOIN catalog cat
  ON cat.track_rowid = c.track_rowid
WHERE c.name_artist = 'Ludwig van Beethoven'
  AND list_contains(cat.tokens, 'op')
  AND list_contains(cat.tokens, '111')
ORDER BY c.track_rowid
""").df()
beethoven_111

,track_rowid,name_track,catalog
0,212690,"Piano Sonata No. 32 in C Minor, Op. 111: I. Ma...",Op. 111
1,816557,"Piano Sonata No. 32 in C Minor, Op. 111: II. A...",Op. 111
2,3048134,"Piano Sonata No. 32 in C Minor, Op. 111: I. Ma...",Op. 111
3,3050839,"Piano Sonata No. 32 in C Minor, Op. 111: I. Ma...",Op. 111
4,3050840,"Piano Sonata No. 32 in C Minor, Op. 111: II. A...",Op. 111
...,...,...,...
992,150871041,"Piano Sonata No. 32 in C Minor, Op. 111: IIb. ...",Op. 111
993,150871042,"Piano Sonata No. 32 in C Minor, Op. 111: IIc. ...",Op. 111
994,150871043,"Piano Sonata No. 32 in C Minor, Op. 111: IId. ...",Op. 111
995,150871044,"Piano Sonata No. 32 in C Minor, Op. 111: IIe. ...",Op. 111


In [ ]:
key_counts = con.execute("""
SELECT
    trim(regexp_replace(
        replace(replace(lower(value), '\u266f', '#'), '\u266d', ' flat '),
        '[^\\p{L}#]+', ' ', 'g'
    )) AS key,
    COUNT(*) AS occurrences
FROM track_tags
WHERE type = 'key'
GROUP BY key
ORDER BY occurrences DESC
""").df()

print(f"distinct keys: {len(key_counts):,}")
key_counts.head(20)

distinct keys: 1,291


,key,occurrences
0,d major,129130
1,c major,110004
2,g major,92045
3,d minor,91096
4,f major,83718
5,e flat major,81083
6,c minor,74618
7,g minor,72565
8,b flat major,72556
9,a major,69492


In [34]:
plot = key_counts.head(12).copy()
plot.loc[len(plot)] = {"key": "other", "occurrences": key_counts["occurrences"].iloc[10:].sum()}

fig = px.pie(
    plot,
    names="key",
    values="occurrences",
    hole=0.45,
    color="key",
    color_discrete_sequence=px.colors.sequential.Teal_r * 2,
    title=f"Key tags (top 12 of {len(key_counts):,})",
)
fig.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hovertemplate="%{label}<br>%{value:,} occurrences (%{percent})<extra></extra>",
)
fig.update_layout(showlegend=False, margin=dict(t=60, b=20, l=20, r=20))
fig.show()

In [35]:
key_pairs = con.execute("""
SELECT DISTINCT
    k.track_rowid,
    k.value AS title_key,
    a."key" AS audio_pitch,
    a.mode AS audio_mode
FROM track_tags k
JOIN (SELECT DISTINCT track_rowid, id_track FROM classical_tracks) c
  ON c.track_rowid = k.track_rowid
JOIN read_parquet('/mnt/titles/classical/track_audio_features.parquet') a
  ON a.track_id = c.id_track
WHERE k.type = 'key'
  AND a."key" IS NOT NULL
  AND a.mode IS NOT NULL
""").df()

print(f"rows: {len(key_pairs):,}")
key_pairs.head()

rows: 1,332,900


,track_rowid,title_key,audio_pitch,audio_mode
0,123196192,B-Flat Minor,6,1
1,5355728,G Minor,7,1
2,24764621,G Minor,6,0
3,24767334,D Minor,0,0
4,24827039,G Major,7,1


In [37]:
PITCH = ["c", "c#", "d", "d#", "e", "f", "f#", "g", "g#", "a", "a#", "b"]
BASE = {"c": 0, "d": 2, "e": 4, "f": 5, "g": 7, "a": 9, "b": 11}

title = (
    key_pairs["title_key"]
    .astype("object")
    .str.lower()
    .str.replace("\u266f", "#", regex=False)
    .str.replace("\u266d", " flat", regex=False)
)

parsed = title.str.extract(r"^([a-g])\s*(#|sharp|flat)?\s*(major|minor)")
parsed.columns = ["tonic", "accidental", "mode_word"]

tonic = parsed["tonic"].astype("object").map(BASE).astype("float")
accidental = parsed["accidental"].astype("object").map({"#": 1, "sharp": 1, "flat": -1}).astype("float").fillna(0)
minor = (parsed["mode_word"] == "minor").astype("int64")

title_index = (tonic + accidental) % 12 + 12 * minor
audio_index = pd.to_numeric(key_pairs["audio_pitch"], errors="coerce") + 12 * (
    pd.to_numeric(key_pairs["audio_mode"], errors="coerce") == 0
)

pairs = pd.DataFrame({"title": title_index, "audio": audio_index}).dropna().astype(int)
print(f"parsed tracks: {len(pairs):,} of {len(key_pairs):,}")

parsed tracks: 993,213 of 1,332,900


In [16]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pairs = pairs.assign(title_pitch=pairs["title"] % 12, title_major=pairs["title"] < 12)

def mode_share(mode):
    is_major = mode == "major"
    sub = pairs[pairs["title_major"] == is_major]
    in_mode = (sub["audio"] < 12) == is_major
    print(f"{mode}: {len(sub):,} tracks, audio labelled {mode}: {in_mode.mean():.1%}")
    counts = pd.crosstab(
        sub.loc[in_mode, "title_pitch"],
        sub.loc[in_mode, "audio"] % 12,
    ).reindex(index=range(12), columns=range(12), fill_value=0)
    return counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0)

major = mode_share("major")
minor = mode_share("minor")

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["titles in major", "titles in minor"],
    horizontal_spacing=0.1,
)
for col, frame in enumerate([major, minor], start=1):
    fig.add_trace(
        go.Heatmap(
            z=frame.values,
            x=[f"{p}" for p in PITCH],
            y=[f"{p}" for p in PITCH],
            colorscale="Teal",
            coloraxis="coloraxis",
            text=frame.values,
            textfont=dict(size=8),
            hovertemplate="title %{y} -> audio %{x}<br>%{z:.2%} of that row<extra></extra>",
        ),
        row=1,
        col=col,
    )

fig.update_xaxes(title_text="audio key", row=1, col=1)
fig.update_xaxes(title_text="audio key", row=1, col=2)
fig.update_yaxes(title_text="title key", row=1, col=1)
fig.update_layout(
    title="Title key vs audio key by mode (rows sum to 100%)",
    coloraxis=dict(colorbar=dict(title="share of row")),
    height=620,
    width=1150,
    margin=dict(t=80, b=40, l=40, r=20),
)
fig.show()

major: 545,405 tracks, audio labelled major: 87.2%
minor: 447,808 tracks, audio labelled minor: 56.1%


In [21]:
bwv_hits = con.execute(r"""
WITH catalog AS (
    SELECT
        track_rowid,
        value,
        regexp_extract_all(lower(value), '[\p{L}\p{N}]+') AS tokens
    FROM track_tags
    WHERE type = 'catalog'
)
SELECT
    track_rowid,
    value,
    try_cast(regexp_extract(lower(value), 'bwv[^0-9]*(\d+)', 1) AS INTEGER) AS bwv
FROM catalog
WHERE array_to_string(tokens, ' ') LIKE '%bwv%'
""").df()

print(f"catalog values with bwv: {len(bwv_hits):,}")
print(f"distinct tracks: {bwv_hits['track_rowid'].nunique():,}")
print(f"without a number: {bwv_hits['bwv'].isna().sum():,}")
bwv_hits.head()

catalog values with bwv: 304,701
distinct tracks: 294,602
without a number: 608


,track_rowid,value,bwv
0,47833727,BWV 785,785
1,24121239,BWV 797,797
2,124902784,BWV 772-786,772
3,124902808,BWV 826,826
4,29379909,BWV 798,798


In [24]:
SECTIONS = [
    (1, 224, "Kantaten"),
    (225, 231, "Motetten"),
    (232, 243, "Messen"),
    (244, 249, "Passionen"),
    (250, 438, "Chor\u00e4le"),
    (439, 524, "Lieder"),
    (525, 771, "Orgel"),
    (772, 994, "Klavier"),
    (995, 1000, "Laute"),
    (1001, 1040, "Kammermusik"),
    (1041, 1071, "Orchester"),
    (1072, 1078, "Kanons"),
    (1079, 1080, "Opfer / Kunst der Fuge"),
]
max_number = max(high for _, high, _ in SECTIONS)

def section_of(number):
    for low, high, name in SECTIONS:
        if low <= number <= high:
            return name
    return "other"

capped = bwv_hits.dropna(subset=["bwv"])
capped = capped[(capped["bwv"] >= 1) & (capped["bwv"] <= max_number)]
print(f"dropped outside 1..{max_number}: {len(bwv_hits.dropna(subset=['bwv'])) - len(capped):,} catalog values")

bwv_counts = (
    capped.groupby("bwv")["track_rowid"]
    .nunique()
    .reset_index(name="tracks")
)
bwv_counts["section"] = bwv_counts["bwv"].map(section_of)

order = [name for _, _, name in SECTIONS]
fig = px.bar(
    bwv_counts,
    x="bwv",
    y="tracks",
    color="section",
    category_orders={"section": order},
    color_discrete_sequence=px.colors.qualitative.Alphabet,
    title=f"Tracks per BWV number ({len(bwv_counts):,} numbers, {bwv_counts['tracks'].sum():,} track-number pairs)",
)
fig.update_traces(marker_line_width=0)
fig.update_layout(
    bargap=0.2,
    height=560,
    width=1200,
    xaxis=dict(title="BWV number", range=[0.5, max_number + 0.5]),
    yaxis=dict(title="tracks (log10)", type="log"),
    legend_title="",
    margin=dict(t=70, b=40, l=50, r=20),
)
fig.show()

dropped outside 1..1080: 1,610 catalog values


In [26]:
version_tokens = con.execute(r"""
SELECT token, COUNT(*) AS occurrences
FROM (
    SELECT unnest(regexp_extract_all(lower(value), '[\p{L}\p{N}]+')) AS token
    FROM track_tags
    WHERE type = 'version'
)
GROUP BY token
ORDER BY occurrences DESC
""").df()

print(f"distinct tokens: {len(version_tokens):,}")
version_tokens.head(20)

distinct tokens: 24,899


,token,occurrences
0,arr,155927
1,live,132848
2,for,132599
3,version,73253
4,by,50782
5,piano,49166
6,and,36284
7,remastered,32868
8,in,23241
9,orchestra,21472


In [27]:
plot = version_tokens.head(10).copy()
plot.loc[len(plot)] = {"token": "other", "occurrences": version_tokens["occurrences"].iloc[10:].sum()}

fig = px.pie(
    plot,
    names="token",
    values="occurrences",
    hole=0.45,
    color="token",
    color_discrete_sequence=px.colors.sequential.Teal_r * 2,
    title=f"Version tag tokens (top 10 of {len(version_tokens):,})",
)
fig.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hovertemplate="%{label}<br>%{value:,} occurrences (%{percent})<extra></extra>",
)
fig.update_layout(showlegend=False, margin=dict(t=60, b=20, l=20, r=20))
fig.show()